<a href="https://colab.research.google.com/github/shafayat19992403/Thesis2/blob/main/PCA_Deflect_3_pcaOnDataSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Function to determine the data root directory**

In [14]:
# !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
# !python rapidsai-csp-utils/colab/env-check.py
# !bash rapidsai-csp-utils/colab/update_gcc.sh
# import os
# os._exit(00)  # This restarts the runtime, continue from next cell afterward
# import condacolab
# condacolab.install()
# !python rapidsai-csp-utils/colab/install_rapids.py stable core
# import os
# os.environ['NUMBAPRO_NVVM'] = '/usr/local/cuda/nvvm/lib64/libnvvm.so'
# os.environ['NUMBAPRO_LIBDEVICE'] = '/usr/local/cuda/nvvm/libdevice/'
# os.environ['CONDA_PREFIX'] = '/usr/local'



# !conda install -c rapidsai -c nvidia -c conda-forge cuml

In [15]:
# import condacolab
# condacolab.install()
# !python rapidsai-csp-utils/colab/install_rapids.py stable core
# import os
# os.environ['NUMBAPRO_NVVM'] = '/usr/local/cuda/nvvm/lib64/libnvvm.so'
# os.environ['NUMBAPRO_LIBDEVICE'] = '/usr/local/cuda/nvvm/libdevice/'
# os.environ['CONDA_PREFIX'] = '/usr/local'



# !conda install -c rapidsai -c nvidia -c conda-forge cuml

In [16]:
# Function to determine the data root directory
import os
def get_data_root():
    if 'COLAB_GPU' in os.environ:
        # Mount Google Drive if needed
        from google.colab import drive
        drive.mount('/content/drive')
        data_root = '/content/drive/MyDrive/PhD/XFED result/Result XFED log/colab output/'
    else:
        data_root = './data/'
    return data_root

# Get the appropriate data root directory
data_root = get_data_root()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ":4096:8"  # or ":16:8"

import sys
import subprocess

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Function to attempt to import a module, and install it if not present
def try_import(module_name, package_name=None):
    try:
        module = __import__(module_name)
        return module
    except ImportError:
        if package_name is None:
            package_name = module_name
        print(f"Installing {package_name}...")
        install(package_name)
        module = __import__(module_name)
        return module

# Standard library imports (no need to install)
import logging
from datetime import datetime
from copy import deepcopy
import gc
import random
import time
import argparse
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable, Union, Optional

# Third-party imports
torch = try_import('torch')
torchvision = try_import('torchvision')
import torchvision.transforms as transforms
import torch.optim as optim
import numpy as np
np = try_import('numpy')
# Import torch.nn as nn
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset, Dataset, ConcatDataset
import torchvision.models as models
from torch.nn.functional import tanh, softmax

from torchvision.datasets import utils
from PIL import Image
import os.path
import shutil


# sklearn imports
sklearn = try_import('sklearn', 'scikit-learn')
from sklearn.model_selection import train_test_split
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances,euclidean_distances
from sklearn.metrics import pairwise_distances
import sklearn.metrics.pairwise as smp
from sklearn.metrics import roc_auc_score

import hdbscan

# Other third-party imports
plt = try_import('matplotlib.pyplot', 'matplotlib')
pd = try_import('pandas')



# torch.use_deterministic_algorithms(True, warn_only=True)
torch.manual_seed(0)

# Device configuration
# Get the number of available GPUs
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {num_gpus}")

# If GPUs are available, choose the desired device index (within the available range)
# Otherwise, default to CPU
if num_gpus > 0:
    desired_gpu_index = 3  # This is the index you originally wanted
    device_index = min(desired_gpu_index, num_gpus - 1)  # Clamp to available range
    device = torch.device(f"cuda:{device_index}")
    torch.cuda.set_device(device)  # Set the device
    print(f"Using GPU: {device}")
else:
    device = torch.device("cpu")
    print("No GPUs available, using CPU.")


#set devices to multiple GPUs
unwanted_device_indices = []
available_device_indices = list(range(num_gpus))
devices = [f'cuda:{i}' for i in available_device_indices if i not in unwanted_device_indices]
if not devices:
    devices = ['cpu']
    # raise RuntimeError("Desired GPUs are not available.")
print(f"Devices: {devices}")




import multiprocessing

# Get the number of available CPU cores
num_cores = multiprocessing.cpu_count()

# Set THREAD_NUMBER to the number of CPU cores
THREAD_NUMBER = min(num_cores, 2*(len(devices)))
# THREAD_NUMBER = 20 # num_cores

print(f"Number of CPU cores available: {num_cores}")
print(f"THREAD_NUMBER set to: {THREAD_NUMBER}")



# Check GPU information
def check_gpu():
    try:
        gpu_info = subprocess.check_output(['nvidia-smi']).decode('utf-8')
        print(gpu_info)
    except Exception as e:
        print('Not connected to a GPU or nvidia-smi not found.')

check_gpu()

# Check CPU information
def check_cpu():
    try:
        cpu_info = subprocess.check_output(['lscpu']).decode('utf-8')
        print(cpu_info)
    except Exception as e:
        print('Could not retrieve CPU information.')

check_cpu()

Number of GPUs available: 0
No GPUs available, using CPU.
Devices: ['cpu']
Number of CPU cores available: 2
THREAD_NUMBER set to: 2
Not connected to a GPU or nvidia-smi not found.
Architecture:                         x86_64
CPU op-mode(s):                       32-bit, 64-bit
Address sizes:                        46 bits physical, 48 bits virtual
Byte Order:                           Little Endian
CPU(s):                               2
On-line CPU(s) list:                  0,1
Vendor ID:                            GenuineIntel
Model name:                           Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                           6
Model:                                79
Thread(s) per core:                   2
Core(s) per socket:                   1
Socket(s):                            1
Stepping:                             0
BogoMIPS:                             4399.99
Flags:                                fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36

# Model Definition for different datasets

In [18]:
class FashionMNISTAlexNet(nn.Module):
    def __init__(self):
        super(FashionMNISTAlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 96, kernel_size=11, stride=4, padding=0),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 10),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

class FeatureNorm(nn.Module):
    def __init__(self, feature_shape):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1, feature_shape))

    def forward(self, x):
        x = torch.einsum('ni, j->ni', x, self.gamma)
        x = x + self.beta
        return  x

class purchase_fully_connected_IN(nn.Module):
    def __init__(self, num_classes):
        super(purchase_fully_connected_IN, self).__init__()
        self.fc1 = nn.Linear(600, 1024, bias=False)  # First layer: input size 600, output size 1024
        self.fc2 = nn.Linear(1024, 100, bias=False)  # Second layer: input size 1024, output size 100
        self.fc3 = nn.Linear(100, num_classes, bias=False)  # Output layer: input size 100, output size num_classes
        self.norm = FeatureNorm(600)

    def forward(self, x):
        x = self.norm(x)
        x = torch.tanh(self.fc1(x))  # Apply tanh activation after the first layer
        x = torch.tanh(self.fc2(x))  # Apply tanh activation after the second layer
        logits = self.fc3(x)         # Output layer, no activation
        return logits

class Purchase(torch.utils.data.Dataset):
    def __init__(self, root =data_root + 'dataset_purchase',train=True, download=True, transform = None):
        self.images = []
        self.root = root
        self.targets = []
        self.train = train
        self.download = download
        self.transform = transform

        x_train, x_test, y_train, y_test = self._train_test_split()

        if self.train:
            self._setup_dataset(x_train, y_train)
        else:
            self._setup_dataset(x_test, y_test)

    def _train_test_split(self):
        df = pd.read_csv(self.root)

        img_names = df.iloc[:, 1:].to_numpy(dtype='f')
        img_label = df.iloc[:, 0].to_numpy()-1
        x_train,x_test, y_train, y_test = train_test_split(img_names, img_label, train_size=0.8,
                                                            random_state=1)
        return x_train, x_test, y_train, y_test

    def _setup_dataset(self, x, y):
            self.images = x
            self.targets = y

    def __len__(self): # Added the __len__ method
        return len(self.images)

    def __getitem__(self, item):
        img = self.images[item]
        label = self.targets[item]
        return img, label

class ThreeLayerDNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ThreeLayerDNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

class FourLayerDNN(nn.Module):
    def __init__(self):
        super(FourLayerDNN, self).__init__()
        # Flatten the input image
        self.flatten = nn.Flatten()
        # Define the fully connected layers
        self.fc1 = nn.Linear(3 * 32 * 32, 1024)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(1024, 512)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(512, 256)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(256, 10)  # Output layer for 10 classes

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.relu3(self.fc3(x))
        x = self.fc4(x)
        return x

class InputNorm(nn.Module):
    def __init__(self, num_channel, num_feature):
        super().__init__()
        self.num_channel = num_channel
        self.gamma = nn.Parameter(torch.ones(num_channel))
        self.beta = nn.Parameter(torch.zeros(num_channel, num_feature, num_feature))
    def forward(self, x):
        if self.num_channel == 1:
            x = self.gamma*x
            x = x + self.beta
            return  x
        if self.num_channel == 3:
            return torch.einsum('...ijk, i->...ijk', x, self.gamma) + self.beta

class mnist_fully_connected_IN(nn.Module):
    def __init__(self,num_classes):
        super(mnist_fully_connected_IN, self).__init__()
        self.hidden1 = 600
        self.hidden2 = 100
        self.fc1 = nn.Linear(28 * 28, self.hidden1, bias=False)
        self.fc2 = nn.Linear(self.hidden1, self.hidden2, bias=False)
        self.fc3 = nn.Linear(self.hidden2, num_classes, bias=False)
        self.relu = nn.ReLU(inplace=False)
        self.norm = InputNorm(1, 28)

    def forward(self,x):
        x = self.norm(x)
        x = x.view(-1, 28 * 28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        logits = self.fc3(x)
        return logits

class CHMNISTDataset(Dataset):
    def __init__(self, image_folder, transform=None):
        self.image_folder = image_folder
        self.transform = transform
        self.image_paths = [os.path.join(image_folder, img) for img in os.listdir(image_folder)]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('L')  # Convert to grayscale
        label = int(image_path.split('_')[-1].split('.')[0])  # Assuming label is in filename

        if self.transform:
            image = self.transform(image)

        return image, label


class HARLogisticRegression(nn.Module):
    """
    A single-layer logistic regression model for multi-class HAR classification.
    """
    def __init__(self, input_dim, num_classes=6):
        super(HARLogisticRegression, self).__init__()
        self.linear = nn.Linear(input_dim, num_classes)  # raw logits

    def forward(self, x):
        return self.linear(x)  # No softmax/sigmoid; use CrossEntropyLoss externally


class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        # Define the linear layer for logistic regression
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        # Apply the linear layer and then the sigmoid activation
        out = torch.sigmoid(self.linear(x))
        return out

class FEMNISTDataset(torchvision.datasets.MNIST):
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False):
        super(torchvision.datasets.MNIST, self).__init__(root, transform=transform, target_transform=target_transform)
        self.download = download
        self.download_link = 'https://media.githubusercontent.com/media/GwenLegate/femnist-dataset-PyTorch/main/femnist.tar.gz'
        self.file_md5 = 'a8a28afae0e007f1acb87e37919a21db'
        self.train = train
        self.root = root
        self.training_file = f'{self.root}/FEMNIST/processed/femnist_train.pt'
        self.test_file = f'{self.root}/FEMNIST/processed/femnist_test.pt'
        self.user_list = f'{self.root}/FEMNIST/processed/femnist_user_keys.pt'

        if not os.path.exists(f'{self.root}/FEMNIST/processed/femnist_test.pt') \
                or not os.path.exists(f'{self.root}/FEMNIST/processed/femnist_train.pt'):
            if self.download:
                self.dataset_download()
            else:
                raise RuntimeError('Dataset not found, set parameter download=True to download')

        if self.train:
            data_file = self.training_file
        else:
            data_file = self.test_file

        data_targets_users = torch.load(data_file)
        self.data, self.targets, self.users = torch.Tensor(data_targets_users[0]), torch.Tensor(data_targets_users[1]), data_targets_users[2]
        self.user_ids = torch.load(self.user_list)

    def __getitem__(self, index):
        img, target = self.data[index], int(self.targets[index])

        # Reshape the flattened image to 28x28
        img = img.view(28, 28).numpy().astype(np.uint8)

        # Convert to PIL Image in grayscale mode
        img = Image.fromarray(img, mode='L')

        if self.transform is not None:
            img = self.transform(img)
        if self.target_transform is not None:
            target = self.target_transform(target)
        return img, target  # Return only img and target

    def dataset_download(self):
        paths = [f'{self.root}/FEMNIST/raw/', f'{self.root}/FEMNIST/processed/']
        for path in paths:
            if not os.path.exists(path):
                os.makedirs(path)

        # download files
        filename = self.download_link.split('/')[-1]
        utils.download_and_extract_archive(self.download_link, download_root=f'{self.root}/FEMNIST/raw/', filename=filename, md5=self.file_md5)

        files = ['femnist_train.pt', 'femnist_test.pt', 'femnist_user_keys.pt']
        for file in files:
            # move to processed dir
            shutil.move(os.path.join(f'{self.root}/FEMNIST/raw/', file), f'{self.root}/FEMNIST/processed/')

Loading Model for different datasets (FashionMNIST, CIFAR-10, PURCHASE, MNIST, EMNIST, CIFAR-100)

In [19]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
class SymbiPredictDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = torch.tensor(self.data[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return features, label

class TabularNet(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TabularNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x


class HARLocalDataset(Dataset):
    """
    A simple PyTorch Dataset for a subset of HAR data (features + labels).
    """
    def __init__(self, features, labels):
        # Convert to PyTorch tensors
        self.features = torch.tensor(features, dtype=torch.float32)

        # Convert labels to numerical if they are not already
        if labels.dtype == np.object_:
            from sklearn.preprocessing import LabelEncoder
            encoder = LabelEncoder()
            labels = encoder.fit_transform(labels)

        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


def load_model(dataset: str):
    """Load and prepare the model and datasets based on the given dataset name."""
    if dataset == 'FashionMNIST':
        transform = transforms.Compose([
            transforms.Resize((227, 227)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        train_data = torchvision.datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
        test_data = torchvision.datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=8)
        classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')
        model = FashionMNISTAlexNet().to(device)

    elif dataset == 'FashionMNIST_3DNN':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        train_data = torchvision.datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
        test_data = torchvision.datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=8)
        classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')
        model = ThreeLayerDNN(input_size=784, hidden_size=512, output_size=10).to(device)

    elif dataset == 'CIFAR10':
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        train_data = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=transform)
        test_data = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=transform)
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=8)
        classes = ('Airplane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck')

        model = models.alexnet(pretrained=True)
        model.classifier[1] = nn.Linear(9216, 4096)
        model.classifier[4] = nn.Linear(4096, 1024)
        model.classifier[6] = nn.Linear(1024, 10)
        model = model.to(device)

    elif dataset == 'PURCHASE':
        train_data = Purchase(train=True, download=True)
        test_data = Purchase(train=False, download=True)
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=8)

        model = purchase_fully_connected_IN(100).to(device)

    elif dataset == 'CHMNIST':
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform) # Assuming CHMNIST is similar to MNIST
        test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
        trainloader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2)
        testloader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2)
        model=models.mobilenet_v2(pretrained=True).to(device)
        model.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)

    elif dataset == 'HAR':
        # Path to your CSV with 563 columns total: 561 features, then 'subject', then 'Activity'
        har_csv_path = os.path.join(data_root, 'har_all_in_one.csv')  # Adjust path as needed

        # Read the CSV. If your CSV has column names as in your screenshot, you can keep header=0
        # But if you have no row of column names, do header=None. Adjust as needed.
        df = pd.read_csv(har_csv_path, header=0)

        # Suppose:
        #   columns [0..560] => 561 features
        #   column 561 => subject in [1..30]
        #   column 562 => activity in [1..6] or string labels
        X = df.iloc[:, :561].values      # shape (N, 561)
        subjects = df.iloc[:, 561].values
        activity = df.iloc[:, 562].values

        # If activity is integer [1..6], but we want [0..5] for CrossEntropyLoss, do:
        # activity = activity - 1  # now [0..5]

        # Standardize all features
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        # We'll build "per-subject" train/test sets
        train_datasets = []
        test_datasets = []

        # Identify unique subject IDs
        unique_subjects = np.unique(subjects)
        print("Subjects found:", unique_subjects)

        for subj_id in unique_subjects:
            # Gather rows for this subject
            subj_mask = (subjects == subj_id)
            X_sub = X[subj_mask]
            y_sub = activity[subj_mask]

            # 75/25 train/test for THIS subject
            X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
                X_sub, y_sub, test_size=0.25, random_state=42
            )

            # Wrap them in Datasets
            ds_train_sub = HARLocalDataset(X_train_sub, y_train_sub)
            ds_test_sub  = HARLocalDataset(X_test_sub,  y_test_sub)

            train_datasets.append(ds_train_sub)
            test_datasets.append(ds_test_sub)

        # Concat all per-subject train sets into one large train_data, likewise for test sets
        train_data = ConcatDataset(train_datasets)
        test_data  = ConcatDataset(test_datasets)

        # Build a testloader for the entire test set
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=4)

        # Build logistic regression model
        input_dim = 561
        # If your activity is 0..5, then num_classes=6
        # If you have 6 distinct string labels, also 6 total classes after label encoding
        num_classes = len(np.unique(activity))
        model = HARLogisticRegression(input_dim, num_classes).to(device)

    elif dataset == 'EMNIST':
        transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

        train_data = torchvision.datasets.EMNIST(root='./data', split='byclass', train=True, download=True, transform=transform)
        test_data = torchvision.datasets.EMNIST(root='./data', split='byclass', train=False, download=True, transform=transform)
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=8)

        model = ThreeLayerDNN(input_size=28 * 28, hidden_size=512, output_size=62).to(device)

    elif dataset == 'MNIST':
        # Define transformation for MNIST
        transform = transforms.Compose([
            transforms.ToTensor(),        # Convert image to PyTorch tensor
            transforms.Normalize((0.5,), (0.5,))  # Normalize grayscale values to [-1, 1]
        ])

        # Load the MNIST dataset ("ByClass" split as an example)
        train_data = torchvision.datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
        test_data = torchvision.datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=8)

        # Load the pre-trained VGG16 model
        model = mnist_fully_connected_IN(10).to(device)

    elif dataset == 'CIFAR100':
        # Install and import CLIP
        try:
            import clip
        except ImportError:
            print("Installing CLIP...")
            install('git+https://github.com/openai/CLIP.git')
            import clip
        # Define the transformation for the dataset (matching CLIP preprocessing)
        transform = transforms.Compose([
            transforms.Resize((224, 224)),  # CLIP expects 224x224 input
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]),
        ])

        # Load the CIFAR100 dataset
        train_data = torchvision.datasets.CIFAR100(root=data_root, train=True, download=True, transform=transform)
        test_data = torchvision.datasets.CIFAR100(root=data_root, train=False, download=True, transform=transform)

        # Create DataLoader for train and test sets
        trainloader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=8)
        testloader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=8)

        # Define the class labels for CIFAR100
        classes = [str(i) for i in range(100)]  # CIFAR100 has 100 classes

        # Load the CLIP model from OpenAI
        model_clip, preprocess = clip.load("ViT-B/32", device=device)

        # Convert CLIP model to float32 to match other layers and data
        model_clip = model_clip.float()

        # Freeze the CLIP model's parameters (we're only training the classifier)
        for param in model_clip.parameters():
            param.requires_grad = False

        # Define a simple 1-layer DNN model on top of CLIP features
        class CLIP_DNN(nn.Module):
            def __init__(self, clip_model, num_classes=100):
                super(CLIP_DNN, self).__init__()
                self.clip_model = clip_model
                self.fc = nn.Linear(512, num_classes)  # CLIP ViT-B/32 gives 512-dimensional features

            def forward(self, images):
                with torch.no_grad():
                    # Extract image features using CLIP's image encoder (cast to float32)
                    image_features = self.clip_model.encode_image(images).float()
                return self.fc(image_features)

        # Initialize the model
        model = CLIP_DNN(model_clip, num_classes=100)

        # Move the model to the device (GPU or CPU)
        model = model.to(device)


    elif dataset == 'SYMBIPREDICT':
        # Load the CSV file
        csv_file = os.path.join(data_root, 'symbipredict_2022.csv')
        df = pd.read_csv(csv_file)

        # Encode target labels
        label_encoder = LabelEncoder()
        df['prognosis'] = label_encoder.fit_transform(df['prognosis'])

        # Separate features and labels
        X = df.drop(columns=['prognosis']).values
        y = df['prognosis'].values

        # Standardize features
        scaler = StandardScaler()
        X = scaler.fit_transform(X)

        # Split into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Create Dataset instances
        train_data = SymbiPredictDataset(X_train, y_train)
        test_data = SymbiPredictDataset(X_test, y_test)

        # DataLoader for test data only
        testloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=4)

        # Define model
        input_dim = X_train.shape[1]
        num_classes = len(label_encoder.classes_)
        model = TabularNet(input_dim=input_dim, num_classes=num_classes).to(device)

    else:
        raise ValueError("Dataset not supported")

    return model, train_data, testloader


In [20]:
def add_trigger(image, trigger_size=15, trigger_value=255):
    # Clone the image to avoid modifying the original one
    triggered_image = image.clone()

    # Add a white square trigger at the bottom-right corner
    triggered_image[:, -trigger_size:, -trigger_size:] = trigger_value / 255.0

    return triggered_image



def evaluate_trigger(testloader, global_model, num_triggered_images_test):
    correct_triggered = 0
    total_triggered = 0
    correct_clean = 0
    total_clean = 0

    with torch.no_grad():
        for idx, data in enumerate(testloader):
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = global_model(images)
            _, predicted = torch.max(outputs.data, 1)

            # Check if the current batch contains any triggered images
            for i in range(len(images)):
                if idx * testloader.batch_size + i < num_triggered_images_test:
                    # Evaluate triggered images
                    total_triggered += 1
                    # print(predicted[i], labels[i])
                    if predicted[i] == labels[i]:
                        correct_triggered += 1
                else:
                    # Evaluate clean images
                    total_clean += 1
                    if predicted[i] == labels[i]:
                        correct_clean += 1

    triggered_accuracy = correct_triggered / total_triggered if total_triggered > 0 else 0
    clean_accuracy = correct_clean / total_clean if total_clean > 0 else 0

    return triggered_accuracy, clean_accuracy

Server Side Methods are here

In [21]:
from typing import List, Tuple, Dict
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from collections import Counter
# from cuml.decomposition import PCA
# from cuml.cluster import DBSCAN



Scalar = Union[bool, bytes, float, int, str, List[int]]
Metrics = Dict[str, Scalar]
# Define metric aggregation function
def weighted_average(metrics: List[Tuple[int, Metrics]]):
    accuracies = [num_examples * m["accuracy"] for num_examples, m in metrics]
    examples = [num_examples for num_examples, _ in metrics]
    return {"accuracy": sum(accuracies) / sum(examples)}

# Define the function to extract client model weights and flatten them
def extract_client_weights(client_models):
    client_weights = []
    for client_model in client_models:  # list of `Parameters` objects
        weights = parameters_to_ndarrays(client_model)  # Convert Parameters to ndarray
        flat_weights = np.concatenate([w.flatten() for w in weights])  # Flatten the weights
        client_weights.append(flat_weights)
    return client_weights


def apply_pca_to_weights(client_weights, client_ids,rnd,flagged_malicious_clients):
    # Apply PCA to reduce to 2 dimensions
    # print(len(client_weights))
    # print(len(client_weights[0]))

    client_weights_np = [w.cpu().numpy() for w in client_weights]  # Move to CPU and convert to NumPy

    client_weights_2d = client_weights_np
    # client_weights_2d = client_weights
    pca = PCA(n_components=2)
    reduced_weights = pca.fit_transform(client_weights_2d)

    # Extract PC1 values
    pc1_values = reduced_weights[:, 0]

    # Reshape PC1 values for clustering
    pc1_values = pc1_values.reshape(-1, 1)

    # Apply DBSCAN clustering based on PC1 values
    #hyperparameter tuning
    if len(flagged_malicious_clients)>0:
        eps_value = 0.15
    else:
        eps_value = 0.2
    dbscan = DBSCAN(eps=eps_value, min_samples=2)  # Adjust eps based on your data
    cluster_labels = dbscan.fit_predict(pc1_values)

    label_counts = Counter(cluster_labels)
    print("Cluster label counts:", label_counts)

    if len(label_counts) > 1:
    # Find the smallest clusters (potential outliers)
        smallest_cluster_size = min(label_counts.values())
        outlier_labels = [label for label, count in label_counts.items() if count == smallest_cluster_size]
        outliers = [client_ids[i] for i, label in enumerate(cluster_labels) if label in outlier_labels]

    else:
        outlier_labels = []
        outliers = []

    # Identify clients belonging to the smallest clusters (potential outliers)


    # Plot the PCA results
    plt.scatter(reduced_weights[:, 0], reduced_weights[:, 1], c=cluster_labels, cmap='viridis', label='Clients')
    plt.colorbar(label='Cluster Label')

    # Annotate clients and highlight outliers
    for i in range(len(client_weights)):
        plt.annotate(f"Client {i}", (reduced_weights[i,0], reduced_weights[i,1]))

    if outliers:
        outlier_indices = [client_ids.index(client_id) for client_id in outliers]
        plt.scatter(reduced_weights[outlier_indices, 0], reduced_weights[outlier_indices, 1], color='red', label='Outliers')


    print("Outliers",outliers)


    plt.title("PCA of Client Weights with DBSCAN-based Outliers (Based on PC1)")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.legend()
    plt.savefig(f"{data_root}/pca_with_dbscan_based_on_pc1_{rnd}_{args.exp_no}.png")
    plt.close()

    #PCA contribution analysis
    # pc1_contributions = np.abs(pca.components_[0])  # Absolute values of the loadings for PC1
    # sorted_contributions = np.sort(pc1_contributions)[::-1]
    # cumulative_contribution = np.cumsum(sorted_contributions)
    # cumulative_percentage = cumulative_contribution / np.sum(pc1_contributions)

    # threshold = 0.2
    # num_weights_90_percent = np.argmax(cumulative_percentage >= threshold) + 1
    # # print(f"Number of weights needed to reach {threshold * 100}% contribution: {num_weights_90_percent}")

    # most_important_weights = np.argsort(pc1_contributions)[::-1][:num_weights_90_percent]
    # most_important_weights_int = [int(weight) for weight in most_important_weights]
    most_important_weights_int = []


    return outliers, most_important_weights_int


In [22]:
def get_predictions_and_labels(net, dataloader):
      all_predictions = []
      all_labels = []
      with torch.no_grad():
          for images, labels in dataloader:
              images, labels = images.to(device), labels.to(device)
              outputs = net(images)
              _, predicted = torch.max(outputs.data, 1)
              all_predictions.extend(predicted.cpu().numpy())
              all_labels.extend(labels.cpu().numpy())
      return np.array(all_predictions), np.array(all_labels)

def test(net):
    """Validate the model on the test set."""
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for images, labels in (triggered_testloader):
            outputs = net(images.to(device))
            labels = labels.to(device)
            loss += criterion(outputs, labels).item()
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
    accuracy = correct / len(triggered_testloader.dataset)
    return loss, accuracy


def evaluate_globalvslocal(global_model, local_history_model):
    # Ensure global and local parameters are set
        if global_model is None or local_history_model is None:
            raise ValueError("Global and local parameters must be set before evaluation.")

        # Set global parameters and evaluate
        # self.set_parameters(self.global_parameters)
        print("Starting Global Testing")
        global_loss, global_accuracy = test(global_model)
        global_predictions, global_labels = get_predictions_and_labels(global_model, triggered_testloader)

        # Set local parameters and evaluate
        print("Starting Local Testing")
        local_loss, local_accuracy = test(local_history_model)
        local_predictions, local_labels = get_predictions_and_labels(local_history_model, triggered_testloader)

        # Calculate False Positive Rate (FPR) for each label
        global_fpr = []
        local_fpr = []
        global_fnr = []
        local_fnr = []
        for label in range(10):  # Assuming 10 classes for MNIST/FashionMNIST
            global_fp = ((global_predictions == label) & (global_labels != label)).sum()
            global_fn = ((global_predictions != label) & (global_labels == label)).sum()
            global_tn = ((global_predictions != label) & (global_labels != label)).sum()
            global_tp = ((global_predictions == label) & (global_labels == label)).sum()
            global_fnr.append(global_fn / (global_fn + global_tp) if (global_fn + global_tp) > 0 else 0)
            global_fpr.append(global_fp / (global_fp + global_tn) if (global_fp + global_tn) > 0 else 0)

            local_fp = ((local_predictions == label) & (local_labels != label)).sum()
            local_tn = ((local_predictions != label) & (local_labels != label)).sum()
            local_fn = ((local_predictions != label) & (local_labels == label)).sum()
            local_tp = ((local_predictions == label) & (local_labels == label)).sum()
            local_fnr.append(local_fn / (local_fn + local_tp) if (local_fn + local_tp) > 0 else 0)
            local_fpr.append(local_fp / (local_fp + local_tn) if (local_fp + local_tn) > 0 else 0)



        # Calculate ROC AUC score per class (one-vs-rest)
        global_roc_auc_per_label = []
        local_roc_auc_per_label = []
        for label in range(10):
            global_roc_auc_label = roc_auc_score((global_labels == label).astype(int), (global_predictions == label).astype(int))
            local_roc_auc_label = roc_auc_score((local_labels == label).astype(int), (local_predictions == label).astype(int))
            global_roc_auc_per_label.append(global_roc_auc_label)
            local_roc_auc_per_label.append(local_roc_auc_label)


        weighted_label_detection = False
        # Save the labels whose local FPR difference is the max compared to global FPR
        if(weighted_label_detection == False):
            max_fpr_diff_label = np.argmax(np.array(local_fpr) - np.array(global_fpr))
            max_fnr_diff_label = np.argmax(np.array(local_fnr) - np.array(global_fnr))
            max_roc_auc_diff_label = np.argmax(np.array(local_roc_auc_per_label) - np.array(global_roc_auc_per_label))
        else:
            fpr_diff = np.array(local_fpr) - np.array(global_fpr)
            roc_auc_diff = np.array(local_roc_auc_per_label) - np.array(global_roc_auc_per_label)
            fnr_diff = np.array(global_fnr) - np.array(local_fnr)
            print(fpr_diff,roc_auc_diff, fnr_diff)
            # scale the difference to 0-1
            fpr_diff = (fpr_diff - np.min(fpr_diff)) / (np.max(fpr_diff) - np.min(fpr_diff))
            roc_auc_diff = (roc_auc_diff - np.min(roc_auc_diff)) / (np.max(roc_auc_diff) - np.min(roc_auc_diff))
            fnr_diff = (fnr_diff - np.min(fnr_diff)) / (np.max(fnr_diff) - np.min(fnr_diff))

            # Write the order of labels in 3 different arrays in descending order
            fpr_diff_sorted_indices = np.argsort(fpr_diff)[::-1]
            roc_auc_diff_sorted_indices = np.argsort(roc_auc_diff)[::-1]
            fnr_diff_sorted_indices = np.argsort(fnr_diff)[::-1]
            with open(f'Figures/ConfigTexts/C{args.cid}_logs.txt', 'a') as f:
                sys.stdout = f
                print("FPR diff sorted labels:", fpr_diff_sorted_indices)
                print("FPR diff values:", fpr_diff[fpr_diff_sorted_indices])
                print("ROC AUC diff sorted labels:", roc_auc_diff_sorted_indices)
                print("ROC AUC diff values:", roc_auc_diff[roc_auc_diff_sorted_indices])
                print("FNR diff sorted labels:", fnr_diff_sorted_indices)
                print("FNR diff values:", fnr_diff[fnr_diff_sorted_indices])
                sys.stdout = sys.__stdout__

            weighted_diff = 0.3 * fpr_diff + 0.3 * roc_auc_diff + 0.4 * fnr_diff

            max_fpr_diff_label = np.argmax(weighted_diff)

        trigger_label = max_fpr_diff_label
        # self.trigger_label = max_roc_auc_diff_label

        # Print global vs local ROC AUC per label
        # for label in range(10):
        #     print(f"Label {label} ---> Global ROC AUC: {global_roc_auc_per_label[label]:.4f}, Local ROC AUC: {local_roc_auc_per_label[label]:.4f}")

        # Plot False Positive Rate (FPR) for each label
        labels = list(range(10))  # Assuming 10 classes for MNIST/FashionMNIST

        plt.figure(figsize=(12, 6))

        # FPR plot
        plt.subplot(1, 3, 1)
        plt.plot(labels, global_fpr, label='Global FPR', marker='o', color='blue')
        plt.plot(labels, local_fpr, label='Local FPR', marker='o', color='green')

        # Highlight points where local FPR is greater than global FPR
        for i, label in enumerate(labels):
            if local_fpr[i] > global_fpr[i]:
                plt.scatter(label, local_fpr[i], color='red', zorder=5, s=100, edgecolor='black', label="Local > Global" if i == 0 else "")

        plt.xlabel('Labels')
        plt.ylabel('False Positive Rate')
        plt.title('FPR Comparison (Global vs Local)')
        plt.legend()

        # ROC AUC plot
        plt.subplot(1, 3, 2)
        plt.plot(labels, global_roc_auc_per_label, label='Global ROC AUC', marker='o', color='blue')
        plt.plot(labels, local_roc_auc_per_label, label='Local ROC AUC', marker='o', color='green')

        # Highlight points where local ROC AUC is greater than global ROC AUC
        for i, label in enumerate(labels):
            if local_roc_auc_per_label[i] > global_roc_auc_per_label[i]:
                plt.scatter(label, local_roc_auc_per_label[i], color='red', zorder=5, s=100, edgecolor='black', label="Local > Global" if i == 0 else "")

        plt.xlabel('Labels')
        plt.ylabel('ROC AUC')
        plt.title('ROC AUC Comparison (Global vs Local)')
        plt.legend()


        # FNR plot
        plt.subplot(1, 3, 3)
        plt.plot(labels, global_fnr, label='Global FNR', marker='o', color='blue')
        plt.plot(labels, local_fnr, label='Local FNR', marker='o', color='green')

        # Highlight points where local FNR is greater than global FNR
        for i, label in enumerate(labels):
            if local_fnr[i] < global_fnr[i]:
                plt.scatter(label, local_fnr[i], color='red', zorder=5, s=100, edgecolor='black', label="Local > Global" if i == 0 else "")

        plt.xlabel('Labels')
        plt.ylabel('False Negative Rate')
        plt.title('FNR Comparison (Global vs Local)')
        plt.legend()


        plt.tight_layout()
        # plt.show()
        plt.savefig(f"{data_root}/_global_vs_local_fpr_roc_auc.png")
        plt.close()
        # trigger_label = 0
        return trigger_label

Aggregation Rules (
FedAvg / Mean,
Median,
Trimmed Mean,
Multi-Krum,
Clipped Clustering,
SignGuard)

Federated Learning Training

In [23]:
def train(loader, optimizer, model, criterion):
  for inputs, targets in loader:
    inputs, targets = inputs.to(device), targets.to(device)
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

  # local_params = torch.cat([param.data.view(-1) for param in model.parameters()])

        # for inputs, targets in sampled_loader:
        #   inputs, targets = inputs.to(device), targets.to(device)
        #   # print(len(inputs), len(targets))
        #   local_optimizer.zero_grad()
        #   local_history_optimizer.zero_grad() if local_history_model is not None else None

        #   outputs = local_model(inputs)
        #   history_outputs = local_history_model(inputs) if local_history_model is not None else None


        #   loss = criterion(outputs, targets)
        #   history_loss = criterion(history_outputs, targets) if local_history_model is not None else None
        #   loss.backward()
        #   history_loss.backward() if local_history_model is not None else None


        #   torch.nn.utils.clip_grad_norm_(local_model.parameters(), max_norm=1.0)
        #   torch.nn.utils.clip_grad_norm_(local_history_model.parameters(), max_norm=1.0) if local_history_model is not None else None

        #   local_optimizer.step()
        #   local_history_optimizer.step()  if local_history_model is not None else None

In [24]:
def train_local_model(client_id, client_indices, global_model, train_data, batch_size, criterion, device, optimizer, client_history_param, triggered_indices):

    # sampled_indices = random.sample(client_indices, min(batch_size, len(client_indices)))
    sampled_indices = random.sample(client_indices, len(client_indices))
    sampled_data = Subset(train_data, sampled_indices)
    # print(f"client_id: {client_id}, sampled_indices: {len(sampled_indices)}, sampled_data: {len(sampled_data)}")
    # sampled_loader = DataLoader(sampled_data, batch_size=len(sampled_indices), shuffle=False, num_workers=0) # Set batch_size to the length of sampled_data
    sampled_loader = DataLoader(sampled_data, batch_size = batch_size, shuffle=False, num_workers=0)

    # Move the model to the assigned GPU device
    local_model = deepcopy(global_model).to(device)
    local_history_model = deepcopy(global_model).to(device) if client_history_param is not None else None

    if client_history_param is not None:
        local_history_model.load_state_dict(client_history_param)


    if optimizer == 'SGD':
        local_optimizer = optim.SGD(local_model.parameters(), lr=0.5, momentum=0.9)
        local_history_optimizer = optim.SGD(local_history_model.parameters(), lr=0.5, momentum=0.9) if local_history_model is not None else None
    else:
        local_optimizer = torch.optim.Adam(local_model.parameters(), lr=0.001)
        local_history_optimizer = torch.optim.Adam(local_history_model.parameters(), lr=0.001) if local_history_model is not None else None


    sampled_loader_filtered, sampled_loader_triggered = None,None
    #trigger label detection
    if client_history_param is not None:
        # trigger_label = evaluate_globalvslocal(local_model,local_history_model)
        targeted_label = 7
        print(f"Trigger label {targeted_label}")

        targeted_images = []
        original_indices = []

        for batch_idx, (images, labels) in enumerate(sampled_loader):
            mask = labels == targeted_label
            targeted_images.append(images[mask])

            # Track original dataset indices of the targeted images
            batch_size = len(labels)
            for i in range(batch_size):
                if mask[i]:
                        global_index = sampled_indices[batch_idx * batch_size + i]
                        original_indices.append(batch_idx * batch_size + i)

        # Define batch size
        batch_size = 16

        # Combine all batches into a single tensor
        targeted_images = torch.cat(targeted_images, dim=0)
        targeted_images = targeted_images.to(device)
        print(f"targeted_images: {len(targeted_images)}")

        features_list = []

        def hook_fn(module, input, output):
            features_list.append(input[0].detach())

        # Register the hook to capture the input to fc2
        handle = local_history_model.fc2.register_forward_hook(hook_fn)

        # Pass the targeted images through the model in batches
        local_history_model.eval()
        with torch.no_grad():
            for i in range(0, len(targeted_images), batch_size):
                batch = targeted_images[i:i + batch_size]
                _ = local_history_model(batch)

        handle.remove()

          # Convert the list of feature outputs to a tensor
        features = torch.cat(features_list, dim=0)

        flattened_features = features.view(features.size(0), -1).cpu().numpy()

        # Apply PCA to reduce to 2 components
        pca = PCA(n_components=2)
        pca_result = pca.fit_transform(flattened_features)
        pc1_values = pca_result[:, 0].reshape(-1, 1)

        # Apply K-means clustering
        kmeans = KMeans(n_clusters=2, random_state=42)
        cluster_labels = kmeans.fit_predict(pc1_values)
        pca_result = np.array(pca_result)
        cluster_labels = np.array(cluster_labels)

        # Identify the sizes of the clusters
        unique, counts = np.unique(cluster_labels, return_counts=True)
        cluster_sizes = dict(zip(unique, counts))

        # Find the smallest cluster
        smallest_cluster_label = max(cluster_sizes, key=cluster_sizes.get)

        # Get indices of samples in the smallest cluster
        smallest_cluster_indices = np.where(cluster_labels == smallest_cluster_label)[0]

        # Convert the smallest cluster indices to their original dataset indices
        smallest_cluster_dataset_indices = [original_indices[i] for i in smallest_cluster_indices]
        indices_to_exclude = set(smallest_cluster_dataset_indices)

                # Compare filtered indices to ground truth
        true_triggered_indices = triggered_indices  # from global context
        true_positives = 0
        false_positives = 0

        for idx in indices_to_exclude:
            # Check if this index in sampled_data corresponds to a triggered index in client's dataset
            global_idx = sampled_indices[idx]  # map local sample index back to global index

            if global_idx in true_triggered_indices:
                true_positives += 1
            else:
                false_positives += 1

        print(f"[Client {client_id}] True Positives: {true_positives}, False Positives: {false_positives} Total Triggers:{len(set(triggered_indices))}")


        local_history_model.eval()
        local_model.eval()

        def create_filtered_trainloader():
                train_dataset = sampled_data
                keep_indices = [i for i in range(len(train_dataset)) if i not in indices_to_exclude]
                filtered_train_dataset = torch.utils.data.Subset(train_dataset, keep_indices)
                triggered_train_dataset = torch.utils.data.Subset(train_dataset, list(indices_to_exclude))

                trainloader_filtered = torch.utils.data.DataLoader(filtered_train_dataset, batch_size=sampled_loader.batch_size, shuffle = False)
                trainloader_triggered = torch.utils.data.DataLoader(triggered_train_dataset, batch_size = sampled_loader.batch_size, shuffle = False)
                return trainloader_filtered, trainloader_triggered

        sampled_loader_filtered, sampled_loader_triggered = create_filtered_trainloader()


        train(sampled_loader_filtered, local_optimizer, local_model, criterion)
        local_params = torch.cat([param.data.view(-1) for param in local_model.parameters()])
        train(sampled_loader_triggered, local_history_optimizer, local_history_model, criterion)
        local_history_params = deepcopy(local_history_model.state_dict())


    else:
        train(sampled_loader, local_optimizer, local_model, criterion)
        local_params = torch.cat([param.data.view(-1) for param in local_model.parameters()])
        local_history_params = None


    # if True:
    #   for inputs, targets in sampled_loader:
    #       inputs, targets = inputs.to(device), targets.to(device)
    #       # print(len(inputs), len(targets))
    #       local_optimizer.zero_grad()
    #       local_history_optimizer.zero_grad() if local_history_model is not None else None

    #       outputs = local_model(inputs)
    #       history_outputs = local_history_model(inputs) if local_history_model is not None else None


    #       loss = criterion(outputs, targets)
    #       history_loss = criterion(history_outputs, targets) if local_history_model is not None else None
    #       loss.backward()
    #       history_loss.backward() if local_history_model is not None else None


    #       torch.nn.utils.clip_grad_norm_(local_model.parameters(), max_norm=1.0)
    #       torch.nn.utils.clip_grad_norm_(local_history_model.parameters(), max_norm=1.0) if local_history_model is not None else None

    #       local_optimizer.step()
    #       local_history_optimizer.step()  if local_history_model is not None else None




    # if True:
    #   train(sampled_loader, local_optimizer, local_model, criterion)
    #   train(sampled_loader, local_history_optimizer, local_history_model, criterion) if local_history_model is not None else None

    #   #Collect model parameters for aggregation
    #   local_params = torch.cat([param.data.view(-1).cpu() for param in local_model.parameters()])
    #   local_params = torch.cat([param.data.view(-1) for param in local_model.parameters()])
    #   local_history_params = deepcopy(local_history_model.state_dict()) if local_history_model is not None else None





    # Cleanup
    del local_model, sampled_data, sampled_loader, sampled_loader_filtered, sampled_loader_triggered
    torch.cuda.empty_cache()
    return client_id, local_params, local_history_params


def federated_learning(num_clients, n_attackers, n_round, batch_size, optim, trigger_label, trigger_fraction, dataset):
    global train_data, testloader
    global triggered_testloader
    """Main federated learning loop."""
    global_model, train_data, testloader = load_model(dataset)

    flagged_malicious_clients = set()
    trust_factor = 0.01;

    criterion = nn.CrossEntropyLoss()
    client_param_history = {}


    # Initialize global_model_data right away:
    global_model_data = torch.cat([p.data.view(-1) for p in global_model.parameters()]).to(device)

    # Step 1: Split the dataset among clients
    total_data_size = len(train_data)
    client_data_size = total_data_size // num_clients
    print("client_data_size", client_data_size)
    indices = list(range(total_data_size))
    random.shuffle(indices)
    clients_data_indices = [indices[i * client_data_size:(i + 1) * client_data_size] for i in range(num_clients)]

    triggered_indices_trainset = [[] for i in range(num_clients)]
    # Adding backdoor using data poisoning attack
    if trigger_label is not None:
        # Convert train_data to a list of tuples to allow modifications
        train_data_list = list(train_data)  # Convert to list of (image, label) tuples

        for local_machine in range(n_attackers):
            print(f"Trigger label for local machine {local_machine} is {trigger_label}")

            triggered_trainset = []
            clean_dataset = []

            num_triggered_images = int(len(clients_data_indices[local_machine]) * trigger_fraction)

            for i in range(num_triggered_images):
                image, label = train_data_list[clients_data_indices[local_machine][i]]
                if label != trigger_label:
                    image = add_trigger(image)
                    label = trigger_label
                    global_index = clients_data_indices[local_machine][i]
                    triggered_indices_trainset[local_machine].append(global_index)

                train_data_list[clients_data_indices[local_machine][i]] = (image, label) # Modify the list

            for i in range(num_triggered_images, len(clients_data_indices[local_machine])):
                image, label = train_data_list[clients_data_indices[local_machine][i]]
                if len(clean_dataset) < 150:
                    clean_dataset.append((image,label))


            print(f"Triggered: {num_triggered_images}")

        train_data = train_data_list
        # print(f"Train Data Size :{len(train_data)}")

        triggered_testset = list(testloader.dataset)
        num_triggered_images_test = int(len(triggered_testset) * trigger_fraction)

        for i in range(num_triggered_images_test):
            if i < num_triggered_images_test:
                image, label = triggered_testset[i]
                if label != trigger_label:
                    image = add_trigger(image)
                    label = trigger_label
                    triggered_testset[i] = (image, label)



        triggered_testloader = DataLoader(triggered_testset, batch_size=64, shuffle=False, num_workers=8)


    #global_models = []

    with ThreadPoolExecutor(max_workers=THREAD_NUMBER) as executor:  # Adjust max_workers based on your system capabilities
        for epoch in range(n_round):
            global_model.train()
            local_models_data_diff = []



            futures = [
                executor.submit(
                    train_local_model,
                    client_id,
                    client_indices,
                    global_model,
                    train_data,
                    batch_size,
                    criterion,
                    devices[client_id % len(devices)],  # Alternate between 'cuda:0' and 'cuda:1'
                    optim,
                    client_param_history.get(client_id, None),
                    set(triggered_indices_trainset[client_id]),

                )
                for client_id, client_indices in enumerate(clients_data_indices)
            ]

            # Collect results
            client_ids = []
            for future in as_completed(futures):
                client_id, local_params, local_history_params = future.result()
                local_models_data_diff.append(local_params)
                client_ids.append(client_id)
                if local_history_params is not None:
                  client_param_history[client_id] = local_history_params

            for i in range(torch.cuda.device_count()):
                torch.cuda.set_device(i)
                torch.cuda.empty_cache()
            if num_gpus > 0:
                torch.cuda.set_device(device)

            print(f'For round {epoch}, training done')
            # time.sleep(30)
            local_models_data = torch.stack(local_models_data_diff).to(device)
            del local_models_data_diff
            gc.collect()

            #finding the malicious clients
            malicious_clients, most_important_weights = apply_pca_to_weights(local_models_data, client_ids, epoch ,flagged_malicious_clients)
            print("Done PCA.....")
            if malicious_clients:
              print(f"Malicious clients detected in round {epoch}: {malicious_clients}")
              print(f"Local Models Stored: {client_param_history.keys()}")
              flagged_malicious_clients.update(malicious_clients)
            else:
              print(f"No malicious clients detected in round {epoch}.")



            #maintaining a malicious client's param list
            for client_id in malicious_clients:
              if client_id not in client_param_history:
                client_param_history[client_id] = global_model.state_dict()



            # Aggregate model updates (f)
            # global_model_data = torch.mean(local_models_data, dim=0)
            if len(flagged_malicious_clients) > 0:
                print("using trust_factor")
                non_malicious_mask = torch.tensor([i not in flagged_malicious_clients for i in range(num_clients)], dtype=torch.bool, device=device)
                local_models_data[~non_malicious_mask] *= trust_factor  # ~ inverts the mask
                global_model_data = torch.sum(local_models_data, dim=0) / (torch.sum(non_malicious_mask) + torch.sum(~non_malicious_mask) * trust_factor)
            else:
                global_model_data = torch.mean(local_models_data, dim=0)
                print("Using normal agg")


            start_idx = 0
            with torch.no_grad():
                for param in global_model.parameters():
                    param_size = param.numel()
                    param.copy_(
                        param.copy_(global_model_data[start_idx:start_idx + param_size].view(param.shape))
                    )
                    start_idx += param_size

            # global_models.append(global_model_data.cpu())
            # global_models.append(global_model_data)

            print(f'For round {epoch}, aggregation done')

            if epoch % 8 == 0 or epoch >= n_round * 0.74:
                # Evaluate global model
                global_model.eval()
                global_model = global_model.to(device)
                correct = 0
                total = 0
                with torch.no_grad():
                    for images, labels in testloader:
                        images, labels = images.to(device), labels.to(device)
                        outputs = global_model(images)
                        _, predicted = torch.max(outputs.data, 1)
                        total += labels.size(0)
                        correct += (predicted == labels).sum().item()

                accuracy = 100 * correct / total
                print(f'Time {datetime.now()}: Accuracy on round {epoch}, total {num_clients}, is: {accuracy:.2f} %')

                # Test backdoor accuracies
                triggered_accuracy, clean_accuracy = evaluate_trigger(triggered_testloader, global_model, num_triggered_images_test)
                print(f'Accuracy on triggered images: {triggered_accuracy * 100:.2f}%')
                print(f'Accuracy on clean images: {clean_accuracy * 100:.2f}%')

            # File path to save the accuracy log
            file_path = os.path.join(data_root, f'accuracy_{dataset}_log.txt')

            # Append accuracy to the file in the data_root location
            with open(file_path, 'a') as f:
                f.write(f'Time {datetime.now()}: Accuracy on round {epoch}, dataset {dataset}, total {num_clients} is: {accuracy:.2f} %\n')

            # global_model = global_model.to('cpu')
            del local_models_data
            torch.cuda.empty_cache()
            gc.collect()


    # Final cleanup after training
    del global_model, train_data, testloader, criterion



In [25]:
class Parameters:
    """Model parameters."""

    tensors: List[bytes]
    tensor_type: str

import numpy.typing as npt
from io import BytesIO
from typing import cast
Any = object()
NDArray = npt.NDArray[Any]
NDArrayInt = npt.NDArray[np.int_]
NDArrayFloat = npt.NDArray[np.float64]
NDArrays = List[NDArray]


def ndarray_to_bytes(ndarray: NDArray) -> bytes:
    """Serialize NumPy ndarray to bytes."""
    bytes_io = BytesIO()
    # WARNING: NEVER set allow_pickle to true.
    # Reason: loading pickled data can execute arbitrary code
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.save.html
    np.save(bytes_io, ndarray, allow_pickle=False)
    return bytes_io.getvalue()


def bytes_to_ndarray(tensor: bytes) -> NDArray:
    """Deserialize NumPy ndarray from bytes."""
    bytes_io = BytesIO(tensor)
    # WARNING: NEVER set allow_pickle to true.
    # Reason: loading pickled data can execute arbitrary code
    # Source: https://numpy.org/doc/stable/reference/generated/numpy.load.html
    ndarray_deserialized = np.load(bytes_io, allow_pickle=False)
    return cast(NDArray, ndarray_deserialized)


def ndarrays_to_parameters(ndarrays: NDArrays) -> Parameters:
    """Convert NumPy ndarrays to parameters object."""
    tensors = [ndarray_to_bytes(ndarray) for ndarray in ndarrays]
    return Parameters(tensors=tensors, tensor_type="numpy.ndarray")


def parameters_to_ndarrays(parameters: Parameters) -> NDArrays:
    """Convert parameters object to NumPy ndarrays."""
    return [bytes_to_ndarray(tensor) for tensor in parameters.tensors]

Example Execution

In [ ]:
# Example execution
# federated_learning(num_clients=200, n_round=300, dataset='EMNIST', batch_size=256, optim="SGD")
# federated_learning(num_clients=40, n_round=250, dataset='FashionMNIST', batch_size=256, optim="SGD")
# federated_learning(num_clients=40, n_round=250, dataset='FashionMNIST_3DNN', batch_size=256, optim="SGD")
parser = argparse.ArgumentParser(description="Federated Learning with Flower and PyTorch")
parser.add_argument("--number_of_round", type=int, default=4, help="Number of rounds")
parser.add_argument("--trust_factor", type=float, default=0.5, help="Trust factor for malicious clients")
parser.add_argument("--exp_no", type=int, default=1, help="Experiment number")
parser.add_argument("--withDefense", type=int, default=1, help="Defense mechanism")
args = parser.parse_args([])
withDefense = args.withDefense == 1

dataset='MNIST'

federated_learning(num_clients=50, n_attackers=10, n_round=50, batch_size=256, optim="SGD", trigger_label=7, trigger_fraction=0.3, dataset="MNIST")
# federated_learning(num_clients=50, n_round=275, dataset='CIFAR10', batch_size=250, optim="SGD")



/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


client_data_size 1200
Trigger label for local machine 0 is 7
Triggered: 360
Trigger label for local machine 1 is 7
Triggered: 360
Trigger label for local machine 2 is 7
Triggered: 360
Trigger label for local machine 3 is 7
Triggered: 360
Trigger label for local machine 4 is 7
Triggered: 360
Trigger label for local machine 5 is 7
Triggered: 360
Trigger label for local machine 6 is 7
Triggered: 360
Trigger label for local machine 7 is 7
Triggered: 360
Trigger label for local machine 8 is 7
Triggered: 360
Trigger label for local machine 9 is 7
Triggered: 360
For round 0, training done
Cluster label counts: Counter({np.int64(1): 40, np.int64(0): 9, np.int64(-1): 1})
Outliers [6]
Done PCA.....
Malicious clients detected in round 0: [6]
Local Models Stored: dict_keys([])
using trust_factor
For round 0, aggregation done


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Time 2025-05-03 11:48:29.342538: Accuracy on round 0, total 50, is: 66.91 %
Accuracy on triggered images: 96.07%
Accuracy on clean images: 69.50%
Trigger label 7
targeted_images: 449
[Client 6] True Positives: 279, False Positives: 28 Total Triggers:323
For round 1, training done


Client side codes are here